In [3]:
## TODO: All project members will need to pip install rasterio 

import torch
import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score

DIR_PATH = "../EuroSATallBands/"

#dataloader class
class EuroSATDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(DIR_PATH + csv_file)

        self.files = df.iloc[:, 0].tolist()
        #make sure the class IDs are parsed as an int
        self.labels = df.iloc[:, 1].astype(int).tolist()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        filename = DIR_PATH + self.files[index]
        #must use rasterio to import the satellite TIF
        with rasterio.open(filename) as img:
            image = img.read().astype(np.float32)

        X = torch.from_numpy(image)
        #this is needed for CrossEntropyLoss later
        Y = torch.tensor(self.labels[index], dtype = torch.int)

        return X, Y

#default parameters: batch size 32, no max, do not shuffle, no workers, random seed of 2 for consistency
def make_dataloader(csv_file, batchsize=32, max_samples = None, shuffle = False, numWorkers = 0, seed = 2):
    r"""
    A function to create a Torch dataloader on a dataset or a subset of a dataset.

    Args:
        csv_file (string): The name of the csv file to be read. Expected format is filepath, classID, className
        batchsize (integer): The size of the batches to load
        max_samples (integer): Default(None) - How many samples to include in the dataloader
        shuffle (boolean): Default(False) - Whether or not the dataloader entires should be shuffled
        numWorkers (integer): Default(0) - How many concurrent workers should be run
        seed(integer): Default(2) - The RNG seed

    """
    dataset = EuroSATDataset(csv_file)
    #This is for splitting the data into smaller subsets so we don't have to train on the whole dataset
    if max_samples is not None:
        if max_samples < 1:
            raise ValueError("Maximum number of records must be positive")

        #for consistency in comparing models
        rng = torch.Generator().manual_seed(seed)
        indices = torch.randperm(dataset.__len__(), generator = rng)[:max_samples].tolist()

        dataset = Subset(dataset, indices)

    return DataLoader(dataset,
                      batch_size = batchsize, 
                      shuffle = shuffle, 
                      num_workers = numWorkers, 
                      pin_memory = torch.cuda.is_available(), 
                      persistent_workers = numWorkers > 0)


In [4]:
# small dataset for model testing
train_loader = make_dataloader(
    "train.csv",
    batchsize=32,
    max_samples=500,
    shuffle=True
)

valid_loader = make_dataloader(
    "validation.csv",
    batchsize=32,
    max_samples=100
)

test_loader = make_dataloader(
    "test.csv",
    batchsize=32,
)
# full dataset
# train_loader = make_dataloader(
#     "train.csv",
#     batchsize=32,
#     shuffle=True
# )

# valid_loader = make_dataloader(
#     "validation.csv",
#     batchsize=32,
# )

# test_loader = make_dataloader(
#     "test.csv",
#     batchsize=32,
# )

In [6]:
#TQDM is here for a built in training progress bar. Put model training below here. 
for X, Y in tqdm(train_loader, desc="Training", unit="batch"):
    pass


def get_labels(dataloader):
    """A function for retrieving the labels from a Dataloader or Subset"""
    dataset = dataloader.dataset
    #if max samples are specified, the dataloader gets a subset, which has a slightly different way to get to the labels
    if isinstance(dataset, Subset):
        labels = np.array(dataset.dataset.labels)[dataset.indices]
    else:
        labels = np.array(dataset.labels)
    return labels



#Model baseline:
Y_Train = get_labels(train_loader)
Y_Test = get_labels(test_loader)
#Scikit baseline doesn't actually use X but needs it specified, so a zero matrix is created
#to avoid loading the images into memory
X_Train = np.zeros((len(Y_Train), 1))
X_Test = np.zeros((len(Y_Test), 1))

baseline = DummyClassifier(strategy="stratified",random_state=2)
baseline.fit(X_Train,Y_Train)
Y_Pred = baseline.predict(X_Test)

print("----[Stratified Random Baseline]----")
print("Accuracy:", accuracy_score(Y_Test, Y_Pred))
print(classification_report(Y_Test,Y_Pred,zero_division=0))

Training:   0%|          | 0/16 [00:00<?, ?batch/s]

----[Stratified Random Baseline]----
Accuracy: 0.10837259876766944
              precision    recall  f1-score   support

           0       0.13      0.12      0.13       300
           1       0.11      0.11      0.11       300
           2       0.12      0.12      0.12       300
           3       0.10      0.11      0.10       250
           4       0.08      0.06      0.07       250
           5       0.07      0.07      0.07       200
           6       0.12      0.14      0.13       250
           7       0.11      0.10      0.10       300
           8       0.08      0.10      0.09       250
           9       0.13      0.13      0.13       359

    accuracy                           0.11      2759
   macro avg       0.11      0.11      0.11      2759
weighted avg       0.11      0.11      0.11      2759



In [ ]:
X, Y = next(iter(train_loader))

print("Images:", X.shape)
print("Labels:", Y.shape)
print("Data type:", X.dtype)




Images: torch.Size([32, 13, 64, 64])
Labels: torch.Size([32])
Data type: torch.float32
